# Notebook 2: Signal Exports

Loads the pre-computed base stack from NB1 and produces all analysis-ready exports:

1. **Multi-scale FRIP** (cross-sectional + annual) at 20 scales × 2 basins
2. **Masked GEDI + covariates** at MODIS resolution per basin

**Prerequisites:** NB1 base stack assets must exist.

**Key design:** Since the base stack is already at ~463m MODIS resolution,
`reduceResolution` from ~463m to 5-100km is trivial (~100-46,000 pixels per output pixel).
`reproject` IS used here to define the output scale for correlation computation,
but the input is a pre-computed asset — not a live fine-resolution chain.

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Asset paths (from NB1)
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study regions (must match NB1)
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Analysis parameters
SCALES = list(range(5000, 105000, 5000))  # 5km to 100km in 5km steps
YEARS = list(range(2001, 2024))
FOREST_COVER_THRESHOLD = 0.95  # >= 95% pristine forest

# GEDI-specific masking thresholds
MAX_ELEVATION = 1000   # meters
MAX_SLOPE = 10         # degrees

print("\u2713 Configuration loaded.")
print(f"  Scales: {SCALES[0]/1000:.0f}km - {SCALES[-1]/1000:.0f}km ({len(SCALES)} scales)")
print(f"  Forest threshold: {FOREST_COVER_THRESHOLD}")

In [ ]:
# =============================================================================
# BLOCK 2: LOAD BASE STACKS
# =============================================================================

def load_base_stack(basin_name):
    """Loads a pre-exported base stack asset from NB1."""
    asset_id = f'{ASSET_ROOT}/BaseStack_{basin_name}'
    return ee.Image(asset_id)

# Verify assets exist
print("Loading base stack assets...")
base_stacks = {}
for basin_name, _ in BASINS:
    try:
        img = load_base_stack(basin_name)
        bands = img.bandNames().getInfo()
        base_stacks[basin_name] = img
        print(f"  \u2713 {basin_name}: {len(bands)} bands loaded")
    except Exception as e:
        print(f"  \u2717 {basin_name}: {e}")
        print(f"    Has NB1 completed? Check: {ASSET_ROOT}/BaseStack_{basin_name}")

if len(base_stacks) == 2:
    print("\n\u2713 Both base stacks available. Ready to compute signals.")
else:
    print("\n\u2717 Missing base stacks. Run NB1 first.")

In [ ]:
# =============================================================================
# BLOCK 3: FRIP COMPUTATION FUNCTIONS
# =============================================================================

def compute_frip_cross(base, scale):
    """Computes cross-sectional FRIP (Spearman correlation between flood
    frequency and median NPP) at the given scale.
    
    Input is a pre-exported ~463m asset, so reduceResolution from ~463m to
    5-100km is computationally trivial. reproject is safe here because
    the input is already materialized.
    """
    base_proj = base.projection()
    
    # Apply pristine forest mask
    masked = base.updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
    )
    
    # Spearman correlation: flood_freq vs Npp_median
    corr = masked.select(['flood_freq', 'Npp_median']).reduceResolution(
        reducer=ee.Reducer.spearmansCorrelation(),
        maxPixels=65535
    ).reproject(crs=base_proj.crs(), scale=scale)
    
    # Mask cells with <10% valid pixel coverage (legacy pattern)
    corr = corr.updateMask(corr.mask().gt(0.1))
    
    return corr.select('correlation').rename(f'FRIP_{scale}')

def compute_frip_annual(base, scale):
    """Computes annual FRIP (23 separate Spearman correlations,
    one per year) at the given scale.
    """
    base_proj = base.projection()
    
    masked = base.updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
    )
    
    def get_annual_corr(year_index):
        year_index = ee.Number(year_index)
        year = ee.Number(2001).add(year_index)
        npp_band = ee.String('NPP_').cat(year.format('%d'))
        
        corr = masked.select([npp_band, 'flood_freq']).reduceResolution(
            reducer=ee.Reducer.spearmansCorrelation(),
            maxPixels=65535
        ).reproject(crs=base_proj.crs(), scale=scale).select('correlation')
        
        corr = corr.updateMask(corr.mask().gt(0.1))
        return corr.set('year', year)
    
    annual_list = ee.List.sequence(0, len(YEARS) - 1).map(get_annual_corr)
    annual_img = ee.ImageCollection.fromImages(annual_list).toBands()
    band_names = [f'FRIP_{y}' for y in YEARS]
    return annual_img.rename(band_names)

print("\u2713 FRIP computation functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 4: GEDI + COVARIATE EXPORT FUNCTION
# =============================================================================

def build_masked_gedi_stack(base):
    """Builds the masked GEDI + covariates stack at MODIS resolution.
    
    Applies both forest and topo masks, then selects the GEDI signal
    bands plus all covariates needed for the adjusted openness model.
    
    Output bands:
      GEDI_UOI, GEDI_N, GEDI_rh98 — signal
      elevation, slope, hnd, precip, clay, forest_fraction — covariates
    """
    # Combined mask: pristine forest AND suitable topography
    mask = (
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
        .And(base.select('elevation').lt(MAX_ELEVATION))
        .And(base.select('slope').lt(MAX_SLOPE))
    )
    
    # Select GEDI signal + covariates and apply mask
    bands = ['GEDI_UOI', 'GEDI_N', 'GEDI_rh98',
             'elevation', 'slope', 'hnd', 'precip', 'clay',
             'forest_fraction']
    
    return base.select(bands).updateMask(mask)

print("\u2713 GEDI export function loaded.")

In [ ]:
# =============================================================================
# BLOCK 5: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running unit tests...\n")
    passed = 0
    
    # Use Congo as test basin
    base = base_stacks.get('Congo')
    if base is None:
        print("  \u2717 No base stack loaded. Run NB1 first.")
        return
    
    # Test 1: Cross-sectional FRIP at 50km
    print("  [1/3] Testing cross-sectional FRIP at 50km...")
    frip = compute_frip_cross(base, 50000)
    bands = frip.bandNames().getInfo()
    assert 'FRIP_50000' in bands, f"Got: {bands}"
    passed += 1
    print(f"    \u2713 Bands: {bands}")
    
    # Test 2: Annual FRIP at 50km (just check bands)
    print("  [2/3] Testing annual FRIP at 50km...")
    frip_annual = compute_frip_annual(base, 50000)
    bands = frip_annual.bandNames().getInfo()
    assert len(bands) == 23, f"Expected 23 annual bands, got {len(bands)}"
    assert 'FRIP_2001' in bands and 'FRIP_2023' in bands, f"Got: {bands}"
    passed += 1
    print(f"    \u2713 {len(bands)} annual bands")
    
    # Test 3: Masked GEDI stack
    print("  [3/3] Testing masked GEDI stack...")
    gedi_stack = build_masked_gedi_stack(base)
    bands = gedi_stack.bandNames().getInfo()
    assert 'GEDI_UOI' in bands, f"Missing GEDI_UOI. Got: {bands}"
    assert 'precip' in bands, f"Missing precip. Got: {bands}"
    passed += 1
    print(f"    \u2713 Bands: {bands}")
    
    print(f"\n{'='*60}")
    print(f"  \u2713 ALL {passed} TESTS PASSED")
    print(f"  Ready to export.")
    print(f"{'='*60}")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 6: EXPORT
# =============================================================================

def safe_start(task, asset_id):
    try:
        ee.data.deleteAsset(asset_id)
        print(f"      Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_frip(dry_run=True):
    """Export multi-scale FRIP (cross-sectional + annual) for both basins.
    
    Produces 80 assets total: 20 scales x 2 types x 2 basins.
    """
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        base = base_stacks[basin_name]
        
        for scale in SCALES:
            # Cross-sectional FRIP
            frip = compute_frip_cross(base, scale)
            cross_id = f'{ASSET_ROOT}/FRIP/FRIP_{scale}_{basin_name}'
            t1 = ee.batch.Export.image.toAsset(
                image=frip,
                description=f'FRIP_{scale}_{basin_name}',
                assetId=cross_id,
                region=basin_geom,
                scale=scale,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((t1, cross_id))
            
            # Annual FRIP
            frip_ann = compute_frip_annual(base, scale)
            ann_id = f'{ASSET_ROOT}/FRIP/FRIP_Annual_{scale}_{basin_name}'
            t2 = ee.batch.Export.image.toAsset(
                image=frip_ann,
                description=f'FRIP_Annual_{scale}_{basin_name}',
                assetId=ann_id,
                region=basin_geom,
                scale=scale,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((t2, ann_id))
    
    print(f"\u2713 FRIP: {len(tasks)} export tasks configured.")
    print(f"  ({len(SCALES)} scales x 2 types x {len(BASINS)} basins)")
    
    if dry_run:
        print("\nDRY RUN. Call export_frip(dry_run=False) to start.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
        print(f"\n\u2713 {len(tasks)} FRIP tasks started!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")

def export_gedi(dry_run=True):
    """Export masked GEDI + covariates at MODIS resolution per basin.
    
    Produces 2 assets: one per basin.
    """
    tasks = []
    base_proj = base_stacks['Congo'].projection()
    
    for basin_name, basin_geom in BASINS:
        base = base_stacks[basin_name]
        gedi_stack = build_masked_gedi_stack(base)
        
        asset_id = f'{ASSET_ROOT}/GEDI/GEDI_masked_{basin_name}'
        task = ee.batch.Export.image.toAsset(
            image=gedi_stack,
            description=f'GEDI_masked_{basin_name}',
            assetId=asset_id,
            region=basin_geom,
            scale=base_proj.nominalScale(),
            crs=base_proj.crs(),
            maxPixels=1e13
        )
        tasks.append((task, asset_id))
    
    print(f"\u2713 GEDI: {len(tasks)} export tasks configured.")
    
    if dry_run:
        print("\nDRY RUN. Call export_gedi(dry_run=False) to start.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
            print(f"  \u2713 Started: {asset_id.split('/')[-1]}")
        print("\n\u2713 GEDI tasks started!")

def export_all(dry_run=True):
    """Export everything: FRIP + GEDI."""
    export_frip(dry_run=dry_run)
    print()
    export_gedi(dry_run=dry_run)

# Show dry run summary
export_all(dry_run=True)